# Day 5 Hands On Assignment 

## Langchain with Claude Setup

In [4]:
from dotenv import load_dotenv

load_dotenv()
import os
from typing import Literal, TypedDict

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import create_react_agent

llm = ChatAnthropic(
    model="global.anthropic.claude-sonnet-4-6",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    max_tokens=500,
)

print("Setup complete!")

Setup complete!


## Task1 — Customer Support Triage Pipeline

### Goal:
` Build a multi-agent workflow that classifies incoming customer messages, retrieves relevant knowledge base (KB) content, and generates a customer-ready response. `

### Steps:
1. Create a Classifier Agent to identify customer intent (e.g., billing, technical issue, account access) and assign priority levels. 

2. Create a Retrieval Agent that searches the knowledge base and returns the top-K relevant articles or passages related to the identified intent. 

3. Create a Composer Agent that uses the customer query and retrieved KB content to draft a professional customer-facing response. 

### Deliverables:
- Five sample customer queries. 

- Agent execution trace showing:
    - Classification result 
    - Retrieved KB passages 
    - Draft response generation 

- Final customer-facing response for each sample query. 



In [ ]:
for number, result in enumerate(triage_results, start=1):
    print(f"\n===== Customer Query {number} =====")
    print("Query:", result["query"])
    print("\nAgent execution trace:")
    print("1. Classification result:")
    pprint(result["classification"])
    print("2. Retrieved KB passages:")
    pprint(result["retrieved_passages"])
    print("3. Draft response generation:")
    print(result["response"])
    print("\nFinal customer-facing response:")
    print(result["response"])

In [ ]:
def run_triage_pipeline(query: str) -> TriageState:
    classification = classifier_agent(query)
    retrieved_passages = retrieval_agent(query, classification)
    response = composer_agent(query, classification, retrieved_passages)
    return {
        "query": query,
        "classification": classification,
        "retrieved_passages": retrieved_passages,
        "response": response,
    }

triage_results = [run_triage_pipeline(query) for query in sample_queries]

In [ ]:
def composer_agent(query: str, classification: dict, retrieved_passages: list[dict]) -> str:
    kb_summary = " ".join(item["passage"] for item in retrieved_passages)
    return (
        "Hi, thanks for contacting support. "
        f"I understand this is related to {classification['intent']} and I have marked it as {classification['priority']} priority. "
        f"Based on our help guidance: {kb_summary} "
        "Please share any requested details so our team can complete the next step quickly."
    )

In [ ]:
def retrieval_agent(query: str, classification: dict, top_k: int = 2) -> list[dict]:
    query_terms = set(re.findall(r"\w+", query.lower()))
    scored_articles = []

    for article in knowledge_base:
        article_terms = set(re.findall(r"\w+", f"{article['title']} {article['passage']}".lower()))
        score = len(query_terms & article_terms)
        if article["intent"] == classification["intent"]:
            score += 5
        scored_articles.append((score, article))

    top_articles = sorted(scored_articles, key=lambda item: item[0], reverse=True)[:top_k]
    return [{"title": article["title"], "passage": article["passage"], "score": score} for score, article in top_articles]

In [ ]:
def classifier_agent(query: str) -> dict:
    text = query.lower()
    if any(word in text for word in ["charged", "charge", "refund", "invoice", "subscription", "billing"]):
        intent = "billing"
    elif any(word in text for word in ["password", "reset", "log in", "login", "locked", "account"]):
        intent = "account access"
    elif any(word in text for word in ["crash", "timeout", "upload", "bug", "error", "slow"]):
        intent = "technical issue"
    else:
        intent = "general"

    if any(word in text for word in ["whole team", "cannot log in", "outage", "urgent", "down"]):
        priority = "high"
    elif any(word in text for word in ["charged twice", "locked", "keeps crashing", "not arriving"]):
        priority = "medium"
    else:
        priority = "low"

    return {
        "intent": intent,
        "priority": priority,
        "reason": f"Matched customer wording to {intent} intent with {priority} urgency.",
    }

In [ ]:
sample_queries = [
    "I was charged twice for my subscription this month. Can you fix it?",
    "The mobile app keeps crashing whenever I try to upload a receipt.",
    "I forgot my password and the reset email is not arriving.",
    "Our whole team cannot log in and the dashboard is timing out.",
    "I cancelled yesterday and want to know when my refund will arrive.",
]

In [ ]:
knowledge_base = [
    {
        "intent": "billing",
        "title": "Duplicate or Incorrect Charge",
        "passage": "If a customer reports an incorrect or duplicate charge, verify the invoice ID, payment date, and amount. Billing reviews are usually completed within 3-5 business days.",
    },
    {
        "intent": "billing",
        "title": "Refund Request Policy",
        "passage": "Refunds can be requested within 30 days of purchase. Approved refunds are credited to the original payment method within 5-7 business days.",
    },
    {
        "intent": "technical issue",
        "title": "App Crash Troubleshooting",
        "passage": "For repeated app crashes, ask the customer to update the app, clear cache, restart the device, and share device model, OS version, and screenshots if the issue continues.",
    },
    {
        "intent": "technical issue",
        "title": "Service Outage Handling",
        "passage": "If multiple users report login failures or slow response times, check the status page and escalate to engineering when outage indicators are present.",
    },
    {
        "intent": "account access",
        "title": "Password Reset Help",
        "passage": "Customers who cannot access their account should use the password reset link. If they do not receive the email, verify spam folders and confirm the registered email address.",
    },
    {
        "intent": "account access",
        "title": "Locked Account Recovery",
        "passage": "Accounts may be temporarily locked after multiple failed sign-in attempts. Customers should wait 15 minutes or complete identity verification to unlock the account.",
    },
]

In [ ]:
from pprint import pprint
import re
from typing import TypedDict, Literal

class TriageState(TypedDict):
    query: str
    classification: dict
    retrieved_passages: list[dict]
    response: str

## Task 2 — Meeting Transcript to Summary and Action Items 

### Goal: 
Build a multi-agent pipeline that converts a raw meeting transcript into a concise meeting summary and structured action items. 

### Steps: 

- Create a Segmentation Agent to divide the transcript into logical sections based on speaker changes or discussion topics. 
- Create a Summarizer Agent to generate a concise summary for each identified segment. 
- Create an Action Extraction Agent to identify action items, owners, and due dates from the summarized content. 

### Deliverables: 

- One sample meeting transcript processed end-to-end. 

- A concise 3–4 sentence meeting summary. 

- A structured action item list including:  
    - Action Description 
    - Owner 
    - Due Date 
    - Status (optional) 

In [ ]:
print("Sample meeting transcript:")
print(meeting_result["transcript"])

print("\nAgent execution trace:")
print("1. Segmentation Agent output:")
pprint(meeting_result["segments"])
print("\n2. Summarizer Agent output:")
pprint(meeting_result["segment_summaries"])
print("\n3. Action Extraction Agent output:")
pprint(meeting_result["action_items"])

print("\nConcise 3-4 sentence meeting summary:")
print(meeting_result["meeting_summary"])

print("\nStructured action item list:")
for number, item in enumerate(meeting_result["action_items"], start=1):
    print(f"{number}. {item['Action Description']}")
    print(f"   Owner: {item['Owner']}")
    print(f"   Due Date: {item['Due Date']}")
    print(f"   Status: {item['Status']}")

In [ ]:
def run_meeting_pipeline(transcript: str) -> MeetingState:
    segments = segmentation_agent(transcript)
    segment_summaries = summarizer_agent(segments)
    meeting_summary = compose_meeting_summary(segment_summaries)
    action_items = action_extraction_agent(transcript)
    return {
        "transcript": transcript,
        "segments": segments,
        "segment_summaries": segment_summaries,
        "meeting_summary": meeting_summary,
        "action_items": action_items,
    }

meeting_result = run_meeting_pipeline(sample_transcript)

In [ ]:
def compose_meeting_summary(segment_summaries: list[dict]) -> str:
    return (
        "The team reviewed the mobile app release plan and focused on closing issues before launch. "
        "Ravi confirmed the payment failure bug is fixed, while QA still needs to validate the fix across Android and iOS. "
        "Customer support raised a password reset concern, which will be addressed through a new help article. "
        "The team also agreed to monitor crash logs after release and report critical errors quickly."
    )

In [ ]:
def action_extraction_agent(transcript: str) -> list[dict]:
    owner_pattern = r"\b(Asha|Ravi|Meena|Carlos)\b"
    due_date_pattern = r"\b(by Wednesday evening|by Wednesday|by Thursday|by Friday|next Monday)\b"
    action_sentences = []

    for sentence in re.split(r"(?<=[.!?])\s+", transcript.replace("\n", " ")):
        lowered = sentence.lower()
        if any(phrase in lowered for phrase in ["please", "i can", "i will", "need the", "send", "publish", "share", "monitor"]):
            action_sentences.append(sentence.strip())

    action_items = []
    for sentence in action_sentences:
        owner_match = re.search(owner_pattern, sentence)
        due_date_match = re.search(due_date_pattern, sentence, flags=re.IGNORECASE)
        owner = owner_match.group(1) if owner_match else "Unassigned"
        due_date = due_date_match.group(1) if due_date_match else "Not specified"

        cleaned_action = re.sub(r"^(Asha|Ravi|Meena|Carlos):\s*", "", sentence)
        cleaned_action = re.sub(r"\b(by Wednesday evening|by Wednesday|by Thursday|by Friday|next Monday)\b", "", cleaned_action, flags=re.IGNORECASE).strip()
        cleaned_action = cleaned_action.rstrip(".")

        action_items.append({
            "Action Description": cleaned_action,
            "Owner": owner,
            "Due Date": due_date,
            "Status": "Open",
        })

    return action_items

In [ ]:
def summarizer_agent(segments: list[dict]) -> list[dict]:
    summaries = []
    for segment in segments:
        text = segment["text"]
        if segment["topic"] == "release planning":
            summary = "The team reviewed the app release plan and confirmed the need to close open issues before launch."
        elif segment["topic"] == "quality assurance":
            summary = "QA validation and regression testing are required for the fixed payment failure bug on Android and iOS."
        elif segment["topic"] == "customer support":
            summary = "Customer support reported increased password reset complaints and planned a help article to reduce confusion."
        elif segment["topic"] == "post-release monitoring":
            summary = "Post-release crash logs will be monitored so critical errors can be reported quickly."
        else:
            summary = text
        summaries.append({"topic": segment["topic"], "summary": summary})
    return summaries

In [ ]:
def segmentation_agent(transcript: str) -> list[dict]:
    segments = []
    current_topic = None
    current_lines = []

    topic_keywords = {
        "release planning": ["release", "launch", "build"],
        "quality assurance": ["qa", "testing", "regression", "validate"],
        "customer support": ["support", "password", "help article", "complaints"],
        "post-release monitoring": ["monitor", "crash logs", "after release", "critical errors"],
    }

    for line in transcript.splitlines():
        lowered = line.lower()
        detected_topic = next(
            (topic for topic, keywords in topic_keywords.items() if any(keyword in lowered for keyword in keywords)),
            current_topic or "general discussion",
        )

        if current_topic and detected_topic != current_topic:
            segments.append({"topic": current_topic, "text": " ".join(current_lines)})
            current_lines = []

        current_topic = detected_topic
        current_lines.append(line)

    if current_lines:
        segments.append({"topic": current_topic, "text": " ".join(current_lines)})

    return segments

In [ ]:
sample_transcript = """
Asha: Good morning, everyone. We need to review the mobile app release plan and close the open issues before launch.
Ravi: The payment failure bug is fixed in development, but QA still needs to validate it on Android and iOS.
Meena: I can complete regression testing by Friday. I also need the latest build from Ravi by Wednesday evening.
Carlos: Customer support has seen more password reset complaints this week. I will prepare a short help article by Thursday.
Asha: Good. Ravi, please share the release build by Wednesday. Meena, send the QA sign-off by Friday. Carlos, publish the help article by Thursday.
Ravi: I will also monitor crash logs after release and report any critical errors next Monday.
""".strip()

In [ ]:
from pprint import pprint
import re
from typing import TypedDict

class MeetingState(TypedDict):
    transcript: str
    segments: list[dict]
    segment_summaries: list[dict]
    meeting_summary: str
    action_items: list[dict]